#### Meta-test split.
20% test data

Group wise split



In [ ]:
import os
import pickle
import numpy as np
from tqdm import tqdm

In [ ]:
files = ['two_class_raw_1s_no.csv', 'two_class_raw_1s_yo_0.5.csv', 'two_class_raw_1s_yo_0.8.csv', 'two_class_raw_2s_no.csv', 'two_class_raw_2s_yo_0.5.csv', 'two_class_raw_2s_yo_0.8.csv', 
        'two_class_raw_3s_no.csv', 'two_class_raw_3s_yo_0.5.csv', 'two_class_raw_3s_yo_0.8.csv', 'two_class_raw_4s_no.csv', 'two_class_raw_4s_yo_0.5.csv', 'two_class_raw_4s_yo_0.8.csv',
        'two_class_raw_5s_no.csv', 'two_class_raw_5s_yo_0.5.csv', 'two_class_raw_5s_yo_0.8.csv']

base_path = '/home/edumaba/Public/MPhil_Thesis/Code/uropatch-data-analysis/improved_features_data_main/'
train_path = '/home/edumaba/Public/MPhil_Thesis/Code/uropatch-data-analysis/working_data'
test_path = '/home/edumaba/Public/MPhil_Thesis/Code/uropatch-data-analysis/meta_test_data'

In [ ]:
from tqdm import tqdm
import os
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

file_results = {}
for file in tqdm(files, desc="Splitting files"):
    data_path = os.path.join(base_path, file)
    features = pd.read_csv(data_path)
    features.drop(['center_time', 'start_time', 'end_time'], axis=1, inplace=True)
    details = file.split('_')
    exp_name = f"{details[3]}_{details[-1].replace('.csv', '')}"
    print(f"Analysing {exp_name}")
    
    # split data
    X = features.drop(columns=['label', 'experiment_id'])
    y = features['label']
    groups = features['experiment_id']

    splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
    train_idx, test_idx = next(splitter.split(X, y, groups))

    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    groups_train, groups_test = groups.iloc[train_idx], groups.iloc[test_idx]

    # Merge back into single DataFrames
    train_df = pd.concat([X_train, y_train, groups_train], axis=1)
    test_df = pd.concat([X_test, y_test, groups_test], axis=1)

    # Save them
    train_file = os.path.join(train_path, f"{exp_name}.csv")
    test_file = os.path.join(test_path, f"{exp_name}.csv")

    train_df.to_csv(train_file, index=False)
    test_df.to_csv(test_file, index=False)

    print(f"Saved: {train_file} and {test_file}")